In [2]:
import sys
import yaml
import argparse
import numpy as np
import igraph as ig
import networkx as nx
import matplotlib.pyplot as plt
from qlgates.config import Config
from qlgates.run_dynamics import propagate_state, build_unitary, bell_state
from qlgates.qlgraphs import qldit, cart_qldit, compute_eigen_decomposition
from qlgates.gates import get_Vg, cnot, transform1
from qlgates.helpers import build_local_operators, expectation
from qlgates.cldyn import transverse_field_ising, initial_state_z_up,evolve_times, propagate_state_classical, transverse_ising_trotter
from qlgates.constants import *
from core.contraction import minimal_quotient
from core.graph_generation import generate_quantum_like_bit

In [3]:
print(sys.executable)   # should show your conda env path, not system python

"""Load YAML and merge into Config dataclass."""
with open("../configs/contraction.yaml") as f:
    overrides = yaml.safe_load(f)  # plain dict from YAML

# These are computed in __post_init__, never let YAML set them
overrides.pop("l", None)
overrides.pop("lp", None)

# Create Config instance with merged parameters
cfg = Config(**overrides)

/Users/sahadebadrita/opt/anaconda3/envs/graphs/bin/python


In [4]:
print(cfg)

Config(debug=False, n=32, k=20, d=1, l=5, lp=4, coupling=True, periodic=False, full=False, NQL=2, CartPdt=True, model='transverse', timesteps=100, deltat=0.1, J=-1.0, h=0.2)


In [5]:
#Checking Will's package
qlbit_1p, info_1p = generate_quantum_like_bit(cfg.n,cfg.k,cfg.l)
qlbit_2p, info_2p = generate_quantum_like_bit(cfg.n,cfg.k,cfg.lp)

In [6]:
#Get Cartesian Product of the full graphs
cartpdt12 = cart_qldit(qlbit_1p,qlbit_2p)
print(cartpdt12.shape)

(4096, 4096)


In [7]:
e_cartpdt12, v_cartpdt12 = np.linalg.eigh(cartpdt12)
print(e_cartpdt12[-5:])

[31.         31.15115352 39.         41.         49.        ]


Test single QL-bit states in min rep

In [8]:
e_qlbit1, v_qlbit1 = np.linalg.eigh(qlbit_1p)
print(e_qlbit1[-3:])

[ 6.10097406 15.         25.        ]


In [12]:
Uz = transform1('z',cfg.n,theta=None,U=None)
print(expectation(v_qlbit1[:,-2:],Uz))

[-1.  1.]


In [13]:
Ux = transform1('x',cfg.n,theta=None,U=None)
print(expectation(v_qlbit1[:,-2:],Ux))

[5.20417043e-16 9.02056208e-17]


Check CNOT gate

In [16]:
UCNOT = cnot(cfg.n, theta=None, U=None)
print(UCNOT.shape)
print(expectation(v_cartpdt12[:,-5:],UCNOT))

(4096, 4096)
[-7.86398941e-16  1.00000000e+00  4.92607257e-16  1.00000000e+00
  1.00000000e+00]


In [18]:
psi_bell_phi_plus = bell_state(cfg, v_cartpdt12[:,-1], kind="phi_plus")

In [20]:
psi_bell_phi_minus = bell_state(cfg, v_min_cartpdt12[:,-1], kind="phi_minus")
print(psi_bell_phi_minus)

[-6.66133815e-16+0.j -7.07106781e-01+0.j -7.07106781e-01+0.j
 -6.10622664e-16+0.j]


In [21]:
psi_bell_psi_plus = bell_state(cfg, v_min_cartpdt12[:,-1], kind="psi_plus")
print(psi_bell_psi_plus)

[-7.07106781e-01+0.j -6.10622664e-16+0.j  6.66133815e-16+0.j
  7.07106781e-01+0.j]


In [22]:
psi_bell_psi_minus = bell_state(cfg, v_min_cartpdt12[:,-1], kind="psi_minus")
print(psi_bell_psi_minus)

[ 6.66133815e-16+0.j  7.07106781e-01+0.j -7.07106781e-01+0.j
 -6.10622664e-16+0.j]


Trotter circuits for Transverse Ising Hamiltonian

In [19]:
#psi0 = v_min_cartpdt12[:,-1]
psi0 = np.kron(v_qlbit1[:,-1],v_qlbit1[:,-1])
psit = propagate_state(cfg,psi0,build_unitary)

In [ ]:
cfg.NQL = 2
times = np.arange(0, cfg.timesteps * cfg.deltat, cfg.deltat)
psi0 = v_qlbit1[:,-1]  # ground state of the first QL-bit
for i in range(cfg.NQL-1):
    psi0 = np.kron(psi0,v_qlbit1[:,-1])  # ground state of the product system
psi0 = psi0 / np.linalg.norm(psi0)  # normalize the state vector

UI = transform1('I',cfg.n,theta=None,U=None)
#U = transform1('z',cfg.n,theta=None,U=None)

for j in range(cfg.NQL):
    mz = build_local_operators(Uz,UI,cfg.NQL)
    print(mz[i].shape)

M_eq_exact = []  # Store equilibrium values for each site
ratios = []  # Store ratios for each site

#for cfg.h in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0]:
for cfg.h in [0.0,0.5,1.0,2.0]:
    #Create Hamiltonian and compute eigenvalues/eigenvectors
    #H = transverse_field_ising(cfg.NQL, cfg.h, cfg.J)
    #print(psi0.shape, H.shape)
    #cfg.h = 0.0
    psit = propagate_state(cfg, psi0, build_unitary)
    print(psit.shape)

    expectation_mz_exact = np.empty((psit.shape[1], cfg.NQL+2))  # +2 for time steps and mean expectation
    for j in range(len(mz)):
        expectation_mz_exact[:,j] = expectation(psit, mz[j])
    expectation_mz_exact[:,-1] = np.arange(cfg.timesteps)  # Add time steps as the last column
    expectation_mz_exact[:,-2] = np.mean(expectation_mz_exact[:,:-2], axis=1)  # Add mean expectation values as the second-to-last column
    M_eq_exact.append(np.mean(np.abs(expectation_mz_exact[int(0.0*cfg.timesteps):,-2])))  # Mean of the absolute values of the mean expectation values in the last 20% of time steps
    ratios.append(abs(cfg.h/cfg.J))  # Ratio of the equilibrium value at this h to the equilibrium value at the smallest h

    print(M_eq_exact)
    print(ratios)


(4096, 4096)
(4096, 4096)
(4096, 100)
[0.9999999999999867]
[0.0]
(4096, 100)
[0.9999999999999867, 0.8756390976008778]
[0.0, 0.2]
(4096, 100)
[0.9999999999999867, 0.8756390976008778, 0.5388329583913417]
[0.0, 0.2, 0.4]


KeyboardInterrupt: 

In [51]:
plt.plot(ratios, M_eq_exact, marker='o')
plt.xlabel('h/|J|')
plt.ylabel(r'$\bar{M}_{z}$')
plt.title(r'$\bar{M}_{z}$ vs h/|J|')
plt.savefig('contraction_NQL10_TFIM.png')
plt.close()
#plt.show()

In [37]:
print(mz)

[array([[ 1.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  1.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j,  0.+0.j, -1.+0.j, -0.+0.j],
       [ 0.+0.j,  0.+0.j, -0.+0.j, -1.+0.j]]), array([[ 1.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
       [ 0.+0.j, -1.+0.j,  0.+0.j, -0.+0.j],
       [ 0.+0.j,  0.+0.j,  1.+0.j,  0.+0.j],
       [ 0.+0.j, -0.+0.j,  0.+0.j, -1.+0.j]])]
